# Función de costo en regresión lineal

La regresión lineal predice una cantidad numérica continua: precio, temperatura, duración, ventas, etc. En este notebook veremos por qué se suele entrenar minimizando el **error cuadrático medio** (*Mean Squared Error*, MSE).

## 1. Predicción y residuo

Con una característica $x$, el modelo es una recta:

$$\hat{y} = wx + b$$

donde $w$ es la pendiente y $b$ el intercepto. Para cada observación, el **residuo** es la diferencia entre el valor real y la predicción:

$$r_i = y_i - \hat{y}_i$$

Si solo sumáramos los residuos, los positivos y negativos se cancelarían. Necesitamos una forma de medir su tamaño sin perder el signo.

In [18]:
import numpy as np
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from scipy.special import huber
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error

datos = pl.DataFrame({
    "horas_estudio": [1, 2, 3, 4, 5, 6],
    "calificacion": [51, 55, 64, 68, 73, 79],
})

w, b = 5.5, 46
datos = datos.with_columns(
    (pl.col("horas_estudio") * w + b).alias("prediccion"),
).with_columns(
    (pl.col("calificacion") - pl.col("prediccion")).alias("residuo"),
)
datos

horas_estudio,calificacion,prediccion,residuo
i64,i64,f64,f64
1,51,51.5,-0.5
2,55,57.0,-2.0
3,64,62.5,1.5
4,68,68.0,0.0
5,73,73.5,-0.5
6,79,79.0,0.0


In [19]:
fig = px.scatter(
    datos, x="horas_estudio", y="calificacion",
    title="Valores reales y recta de regresión",
)
fig.add_trace(go.Scatter(
    x=datos["horas_estudio"], y=datos["prediccion"],
    mode="lines", name="predicción",
))
for fila in datos.iter_rows(named=True):
    fig.add_shape(
        type="line", x0=fila["horas_estudio"], x1=fila["horas_estudio"],
        y0=fila["prediccion"], y1=fila["calificacion"],
        line={"dash": "dot", "color": "gray"},
    )
fig.show()

## 2. Error cuadrático medio (MSE)

La función de costo más habitual eleva cada residuo al cuadrado y calcula su promedio:

$$J(w,b) = \frac{1}{n} \sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

Sustituyendo $\hat{y}_i = wx_i + b$:

$$J(w,b) = \frac{1}{n} \sum_{i=1}^{n}(y_i - (wx_i+b))^2$$

Al cuadrar logramos tres cosas: los errores no se cancelan, los errores grandes reciben una penalización mayor y la función es suave y derivable. La raíz del MSE es el **RMSE**, que vuelve a las unidades de la variable objetivo.

In [20]:
costo_mse = mean_squared_error(datos["calificacion"], datos["prediccion"])
costo_rmse = root_mean_squared_error(datos["calificacion"], datos["prediccion"])
print(f"MSE:  {costo_mse:.3f}")
print(f"RMSE: {costo_rmse:.3f} puntos de calificación")

datos.with_columns((pl.col("residuo") ** 2).alias("residuo_al_cuadrado"))

MSE:  1.125
RMSE: 1.061 puntos de calificación


horas_estudio,calificacion,prediccion,residuo,residuo_al_cuadrado
i64,i64,f64,f64,f64
1,51,51.5,-0.5,0.25
2,55,57.0,-2.0,4.0
3,64,62.5,1.5,2.25
4,68,68.0,0.0,0.0
5,73,73.5,-0.5,0.25
6,79,79.0,0.0,0.0


## 3. MAE, RMSE y Huber Loss: distintas prioridades

El **MAE** (*Mean Absolute Error*, error absoluto medio) calcula el promedio de las distancias entre realidad y predicción, sin elevarlas al cuadrado:

$$MAE = \frac{1}{n}\sum_{i=1}^{n}|y_i-\hat{y}_i|$$

Es fácil de explicar: un MAE de 4 en una predicción de minutos significa que, en promedio, el modelo se equivoca por 4 minutos. Un error de 10 cuenta exactamente el doble que uno de 5. Por eso MAE es menos sensible a valores atípicos que MSE. Es una buena elección cuando los errores grandes son posibles pero no deberían dominar el modelo; por ejemplo, al estimar tiempos de entrega, precios típicos de alquiler o demanda habitual.

El **RMSE** (*Root Mean Squared Error*) es simplemente la raíz del MSE:

$$RMSE = \sqrt{MSE}$$

MSE y RMSE prefieren exactamente los mismos parámetros; la raíz cuadrada no cambia qué recta es la mejor. La ventaja del RMSE es que vuelve a las unidades originales: si la variable es una calificación, el RMSE está expresado en puntos de calificación. Úsalo cuando los errores grandes tienen un costo especialmente alto —por ejemplo, subestimar mucho la demanda de ambulancias— y quieres que el modelo se concentre en reducirlos.

La **Huber Loss** es una alternativa cuando pueden existir valores atípicos. Para residuos pequeños se comporta como el error cuadrático (es suave y preciso); a partir de un umbral $\delta$, crece de forma lineal y evita que un único error enorme domine el aprendizaje:

$$L_\delta(r) = \begin{cases} \frac{1}{2}r^2 & |r|\leq\delta \\ \delta(|r|-\frac{1}{2}\delta) & |r|>\delta \end{cases}$$

No hay una métrica Huber en `sklearn.metrics`, así que usaremos `scipy.special.huber`, una implementación numérica ya probada de SciPy; no recreamos la función a mano.

In [21]:
# Comparamos el conjunto normal con otro que contiene una observación extrema.
datos_con_atipico = pl.concat([
    datos,
    pl.DataFrame({"horas_estudio": [7], "calificacion": [20], "prediccion": [84.5], "residuo": [-64.5]}),
])

y_sin_atipico = datos["calificacion"].to_numpy()
pred_sin_atipico = datos["prediccion"].to_numpy()
y_con_atipico = datos_con_atipico["calificacion"].to_numpy()
pred_con_atipico = datos_con_atipico["prediccion"].to_numpy()

pl.DataFrame({
    "conjunto": ["sin valor atípico", "con valor atípico"],
    "MAE": [
        mean_absolute_error(y_sin_atipico, pred_sin_atipico),
        mean_absolute_error(y_con_atipico, pred_con_atipico),
    ],
    "MSE": [
        mean_squared_error(y_sin_atipico, pred_sin_atipico),
        mean_squared_error(y_con_atipico, pred_con_atipico),
    ],
    "RMSE": [
        root_mean_squared_error(y_sin_atipico, pred_sin_atipico),
        root_mean_squared_error(y_con_atipico, pred_con_atipico),
    ],
    "Huber promedio (delta=5)": [
        np.mean(huber(5.0, y_sin_atipico - pred_sin_atipico)),
        np.mean(huber(5.0, y_con_atipico - pred_con_atipico)),
    ],
})

conjunto,MAE,MSE,RMSE,Huber promedio (delta=5)
str,f64,f64,f64,f64
"""sin valor atípico""",0.75,1.125,1.06066,0.5625
"""con valor atípico""",9.857143,595.285714,24.398478,44.767857


El dato extremo hace subir mucho MSE y RMSE porque su residuo se eleva al cuadrado. MAE también aumenta, pero de forma proporcional al error. Huber también lo penaliza —no lo ignora—, pero después de $\delta=5$ su castigo crece linealmente. Elegir $\delta$ depende de la escala de la variable y de cuánto error consideres normal.

En un proyecto real, empieza comparando MAE y RMSE: si RMSE es mucho mayor que MAE, probablemente existan algunos errores grandes. Considera Huber cuando tengas razones para pensar que hay errores de medición o valores extremos que no deben decidir por sí solos la recta.

### Caso real: estimar tiempos de entrega de comida

Una aplicación predice los minutos que tardará un pedido. Cinco pedidos normales tienen errores pequeños, pero uno sufrió un accidente de tráfico y tardó 150 minutos más de lo previsto. Ese evento es real, pero no representa una entrega habitual.

Calcularemos las cuatro medidas sobre el **mismo modelo**. No buscamos decidir cuál número es universalmente mejor: queremos que la elección dependa de la decisión del negocio.

In [26]:
# Minutos reales y estimados por un mismo modelo de entregas.
entregas = pl.DataFrame({
    "pedido": ["P1", "P2", "P3", "P4", "P5", "P6: accidente"],
    "minutos_reales": [28, 35, 42, 50, 60, 55],
    "minutos_predichos": [30, 39, 44, 47, 56, 50],
}).with_columns(
    (pl.col("minutos_reales") - pl.col("minutos_predichos")).alias("residuo"),
)

y_real = entregas["minutos_reales"].to_numpy()
y_predicho = entregas["minutos_predichos"].to_numpy()
delta = 15.0  # Consideramos 15 minutos como el límite de un error habitual.

metricas_entregas = pl.DataFrame({
    "medida": ["MAE", "MSE", "RMSE", "Huber promedio (delta=15)"],
    "valor": [
        mean_absolute_error(y_real, y_predicho),
        mean_squared_error(y_real, y_predicho),
        root_mean_squared_error(y_real, y_predicho),
        np.mean(huber(delta, y_real - y_predicho)),
    ],
})
entregas, metricas_entregas

(shape: (6, 4)
 ┌───────────────┬────────────────┬───────────────────┬─────────┐
 │ pedido        ┆ minutos_reales ┆ minutos_predichos ┆ residuo │
 │ ---           ┆ ---            ┆ ---               ┆ ---     │
 │ str           ┆ i64            ┆ i64               ┆ i64     │
 ╞═══════════════╪════════════════╪═══════════════════╪═════════╡
 │ P1            ┆ 28             ┆ 30                ┆ -2      │
 │ P2            ┆ 35             ┆ 39                ┆ -4      │
 │ P3            ┆ 42             ┆ 44                ┆ -2      │
 │ P4            ┆ 50             ┆ 47                ┆ 3       │
 │ P5            ┆ 60             ┆ 56                ┆ 4       │
 │ P6: accidente ┆ 55             ┆ 50                ┆ 5       │
 └───────────────┴────────────────┴───────────────────┴─────────┘,
 shape: (4, 2)
 ┌───────────────────────────┬───────────┐
 │ medida                    ┆ valor     │
 │ ---                       ┆ ---       │
 │ str                       ┆ f64       │
 ╞═══

Cómo leer el resultado:

- **MAE** responde: “en una entrega típica, ¿cuántos minutos me equivoco?”. Es útil para informar una expectativa comprensible al cliente y comparar modelos de operación normal. El accidente aporta 145 minutos de error, pero no aplasta a los otros cinco pedidos.
- **MSE** multiplica el efecto del accidente al elevar 145 al cuadrado. Sirve como función de entrenamiento si un retraso enorme tiene un costo desproporcionado: multas por SLA, comida desperdiciada o pérdida grave de clientes. Su unidad es minutos cuadrados, por eso no es intuitiva para comunicar.
- **RMSE** mantiene esa prioridad sobre errores graves, pero se expresa en minutos. Es una forma más comunicable de reportar el comportamiento que MSE está optimizando.
- **Huber Loss** se comporta como MSE para los cinco pedidos normales —motiva a afinarlos— y pasa a crecer linealmente para el accidente. Es razonable si el accidente es excepcional y no quieres que el modelo empiece a sobrepredecir todos los pedidos para protegerse de eventos raros.

En resumen: usa MAE para el error típico, MSE/RMSE cuando los fallos grandes son especialmente costosos y Huber cuando quieres precisión normal sin dejar que unos pocos casos anómalos gobiernen el ajuste. Siempre valida la elección con el costo real del problema, no solo con el valor más pequeño de una métrica.

## 4. El costo como superficie

Cada combinación de pendiente $w$ e intercepto $b$ produce un MSE distinto. En regresión lineal, esa superficie tiene forma de cuenco convexo: existe un mínimo global. Esto es importante porque, a diferencia de otros modelos, no hay mínimos locales que puedan atrapar al optimizador.

In [23]:
x = datos["horas_estudio"].to_numpy()
y = datos["calificacion"].to_numpy()
pendientes = np.linspace(2, 9, 70)
interceptos = np.linspace(35, 58, 70)

superficie = pl.DataFrame({
    "pendiente": np.repeat(pendientes, len(interceptos)),
    "intercepto": np.tile(interceptos, len(pendientes)),
}).with_columns(
    pl.struct(["pendiente", "intercepto"]).map_elements(
        lambda fila: mean_squared_error(y, fila["pendiente"] * x + fila["intercepto"]),
        return_dtype=pl.Float64,
    ).alias("mse")
)

px.density_heatmap(
    superficie, x="pendiente", y="intercepto", z="mse",
    histfunc="avg", color_continuous_scale="Viridis_r",
    title="Superficie de costo: el mínimo es la zona más oscura",
).show()

## 5. Gradientes y descenso de gradiente

Para reducir el costo, calculamos cómo cambia respecto a cada parámetro:

$$\frac{\partial J}{\partial w} = -\frac{2}{n}\sum_{i=1}^{n}x_i(y_i-\hat{y}_i), \qquad \frac{\partial J}{\partial b} = -\frac{2}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)$$

El descenso de gradiente actualiza en dirección contraria al gradiente:

$$w \leftarrow w - \alpha \frac{\partial J}{\partial w}, \qquad b \leftarrow b - \alpha \frac{\partial J}{\partial b}$$

donde $\alpha$ es la tasa de aprendizaje.

In [24]:
w, b = 0.0, 0.0
tasa_aprendizaje = 0.03
historial = []

for epoca in range(300):
    pred = w * x + b
    error = y - pred
    grad_w = -2 * np.mean(x * error)
    grad_b = -2 * np.mean(error)
    w -= tasa_aprendizaje * grad_w
    b -= tasa_aprendizaje * grad_b
    historial.append({"epoca": epoca + 1, "mse": mean_squared_error(y, w * x + b)})

print(f"Parámetros aprendidos: w={w:.3f}, b={b:.3f}")
px.line(pl.DataFrame(historial), x="epoca", y="mse", title="El MSE disminuye durante el entrenamiento").show()

Parámetros aprendidos: w=6.015, b=43.669


## 6. Ideas clave

- El MSE es el promedio de los residuos al cuadrado.
- Penaliza de forma fuerte los errores grandes, por lo que puede ser sensible a valores atípicos.
- Bajo el supuesto de errores normales con varianza constante, minimizar MSE equivale a máxima verosimilitud.
- El RMSE es más fácil de interpretar porque conserva las unidades de $y$.
- Huber Loss conserva la sensibilidad a errores pequeños, pero reduce la influencia de errores extremos.
- Si hay valores atípicos importantes, compara también con MAE (*Mean Absolute Error*) o modelos robustos.

**Ejercicio:** añade un dato extremo, por ejemplo 7 horas y calificación 20. Compara cómo cambian MSE y MAE, y observa el desplazamiento de la recta.